# تحلیل دادهٔ شکایت‌های گرمایش NYC 311

این نوت‌بوک خروجی‌های محلی کارخانه را بررسی می‌کند و برای اجرا به API نیاز ندارد. ورودی تحلیل، `analytical_records.csv` و `monthly_summary_wide.csv` است؛ بنابراین نتایج فقط زمانی به‌روز هستند که این فایل‌ها از اجرای curation و تحلیل ساخته شده باشند.

دامنه شامل درخواست‌های `HEAT/HOT WATER` از ۱ ژانویهٔ ۲۰۲۴ تا پیش از ۱ ژوئیهٔ ۲۰۲۵ است. هدف، توصیف الگوهای ثبت و زمان بسته‌شدن است، نه استنباط علت یا ارزیابی عملکرد یک سازمان. نرخ‌ها نیز به‌دلیل استفاده از ZCTA به‌عنوان تقریب ZIP باید با احتیاط تفسیر شوند.

## ۱. بارگذاری و کنترل خروجی‌ها

همهٔ مسیرها از ریشهٔ پروژه خوانده می‌شوند. شمار رکوردها با quality report مقایسه می‌شود و تعداد تطبیق‌های جمعیتی و اختلاف borough دوباره از دادهٔ رکوردی محاسبه می‌شود.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from nyc311_factory.analytics import load_population
from nyc311_factory.config import load_settings

ROOT = Path.cwd()
settings = load_settings(ROOT / "config" / "project.json")
curated_dir = ROOT / "data" / "curated"
analysis_path = curated_dir / "analytical_records.csv"
summary_path = curated_dir / "monthly_summary_wide.csv"
quality_path = curated_dir / "quality_report.json"

for path in (analysis_path, summary_path, quality_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required factory output is missing: {path}")

records = pd.read_csv(
    analysis_path,
    dtype={"unique_key": "string", "incident_zip": "string"},
    low_memory=False,
)
records["created_date"] = pd.to_datetime(records["created_date"], utc=True, errors="raise")
records["closure_days"] = pd.to_numeric(records["closure_days"], errors="coerce")
records["population"] = pd.to_numeric(records["population"], errors="coerce")
records["month"] = records["created_date"].dt.strftime("%Y-%m")
monthly_summary = pd.read_csv(summary_path, index_col="borough")
quality_report = json.loads(quality_path.read_text(encoding="utf-8"))
population = load_population(settings.population_path)

if len(records) != quality_report["accepted_records"]:
    raise ValueError("Analytical row count differs from the curation quality report")

borough_comparable = records["borough"].notna() & records["reference_borough"].notna()
borough_disagreements = int(
    (borough_comparable & (records["borough"] != records["reference_borough"])).sum()
)
print(f"Analytical records: {len(records):,}")
print(f"Distinct request keys: {records['unique_key'].nunique():,}")
print(f"Population matched: {int(records['population_matched'].sum()):,}")
print(f"Population unmatched: {int((~records['population_matched']).sum()):,}")
print(f"311/reference borough disagreements: {borough_disagreements:,}")
print(f"Population reference ZIP rows: {len(population):,}")
print(f"Date range in output: {records['created_date'].min()} to {records['created_date'].max()}")

## ۲. آمار توصیفی و بررسی مقادیر دورافتاده

زمان بسته‌شدن برای پرونده‌های باز تهی است. نمودارهای مدت، پرونده‌های بسته‌شده را نشان می‌دهند. مرز ۱٫۵ برابر دامنهٔ بین چارکی فقط یک قاعدهٔ علامت‌گذاری آماری است؛ به‌تنهایی دلیل حذف یک رکورد معتبر نیست.

In [ ]:
closure = records["closure_days"].dropna()
q1 = closure.quantile(0.25)
q3 = closure.quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
p99 = closure.quantile(0.99)

print(f"Open requests: {records['closure_days'].isna().sum():,}")
print(f"Closed requests: {len(closure):,}")
print(f"IQR upper fence: {upper_fence:.3f} days")
print(f"Closed requests above the fence: {(closure > upper_fence).sum():,}")
print(f"Requests above the 99th percentile: {(closure > p99).sum():,}")
closure.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(3)

## ۳. حجم درخواست‌ها در طول زمان

نمودار تعداد رکوردها در هر ماه به تفکیک borough است. تعداد درخواست‌ها، نه تعداد افراد یا ساختمان‌های یکتا، اندازه‌گیری می‌شود.

In [ ]:
borough_monthly_counts = (
    records.dropna(subset=["borough"])
    .groupby(["month", "borough"]).size()
    .unstack("borough", fill_value=0)
)

fig, ax = plt.subplots(figsize=(12, 5))
borough_monthly_counts.plot(ax=ax, marker="o", linewidth=1.8)
ax.set(title="Monthly HEAT/HOT WATER requests by borough", xlabel="Month", ylabel="Requests")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="311 borough", ncols=2)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

در این برش، اوج ماهانه در ژانویهٔ ۲۰۲۵ و کمینه در اوت ۲۰۲۴ دیده می‌شود. افزایش زمستانی با موضوع گرمایش سازگار است، اما به‌تنهایی اثبات‌کنندهٔ فصل‌مندی علّی نیست؛ تغییر رفتار ثبت یا ویژگی‌های منبع نیز می‌تواند بر شمارش اثر بگذارد.

## ۴. توزیع زمان بسته‌شدن

برای خوانایی، محور افقی نمودار در صدک ۹۹ متوقف می‌شود؛ دنبالهٔ بالاتر از آن حذف نشده و تعدادش روی نمودار گزارش می‌شود.

In [ ]:
visible_closures = closure[closure <= p99]
tail_count = int((closure > p99).sum())
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(visible_closures, bins=50, color="steelblue", edgecolor="white")
ax.axvline(closure.median(), color="darkorange", linestyle="--", label=f"Median: {closure.median():.2f} days")
ax.axvline(p99, color="darkred", linestyle=":", label=f"99th percentile: {p99:.2f} days")
ax.set(title="Closure duration (view capped at the 99th percentile)", xlabel="Days from creation to closure", ylabel="Requests")
ax.text(0.98, 0.95, f"Above axis cap: {tail_count:,}", transform=ax.transAxes, ha="right", va="top")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

توزیع به‌شدت راست‌چوله است: میانه حدود ۱٫۵۷ روز است، صدک ۹۹ حدود ۵٫۰۴ روز، ولی بیشینه بسیار دورتر قرار دارد. مقدارهای بلندمدت با این داده‌ها به‌تنهایی خطای ثبت محسوب نمی‌شوند؛ به همین دلیل در خروجی نگه داشته می‌شوند و فقط در نمایش‌های فشرده پنهان می‌شوند.

## ۵. مقایسهٔ زمان بسته‌شدن میان boroughها

نقاط دورافتاده در نمودار جعبه‌ای برای خوانایی نمایش داده نمی‌شوند، اما در محاسبه و فایل تحلیلی باقی می‌مانند.

In [ ]:
boroughs = ["BRONX", "BROOKLYN", "MANHATTAN", "QUEENS", "STATEN ISLAND"]
closure_by_borough = [
    records.loc[records["borough"] == borough, "closure_days"].dropna()
    for borough in boroughs
]
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(closure_by_borough, tick_labels=boroughs, showfliers=False)
ax.set(title="Closure duration by 311 borough", xlabel="311 borough", ylabel="Days")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

borough_closure_stats = records.groupby("borough").agg(
    requests=("unique_key", "size"),
    median_closure_days=("closure_days", "median"),
    open_requests=("closure_days", lambda values: int(values.isna().sum())),
)
print(borough_closure_stats.round(3).to_string())

میانهٔ زمان بسته‌شدن در Bronx از Manhattan پایین‌تر است. این مقایسهٔ توصیفی است و با شدت یا نوع درخواست، شرایط عملیاتی، و ترکیب پرونده‌ها تعدیل نشده است؛ بنابراین از آن نباید دربارهٔ علت یا عملکرد نسبی نتیجه‌گیری کرد.

## ۶. نرخ ماهانه نسبت به جمعیت تقریبی

مخرج نرخ، جمعیت ZIP/ZCTAهای مرجع منتسب به هر borough است. نرخ، شمار درخواست به ازای ۱۰٬۰۰۰ نفر را نشان می‌دهد و شمار افراد یکتا نیست.

In [ ]:
months = sorted(records["month"].unique())
rate_columns = [f"rate_per_10k_{month}" for month in months]
rates_by_month = monthly_summary.reindex(boroughs)[rate_columns].T
rates_by_month.index = pd.to_datetime(months, format="%Y-%m")
fig, ax = plt.subplots(figsize=(12, 5))
rates_by_month.plot(ax=ax, marker="o", linewidth=1.8)
ax.set(title="Monthly HEAT/HOT WATER requests per 10,000 residents", xlabel="Month", ylabel="Requests per 10,000")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="311 borough", ncols=2)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

count_columns = [f"count_{month}" for month in months]
period_rates = (monthly_summary[count_columns].sum(axis=1) / monthly_summary["borough_population"] * 10_000)
print("18-month cumulative requests per 10,000 reference residents:")
print(period_rates.reindex(boroughs).round(1).to_string())

## ۷. رابطهٔ حجم ماهانه و میانهٔ زمان بسته‌شدن

هر نقطه یک جفت borough-ماه است. ضریب Spearman با هم‌بستگی Pearson روی رتبه‌ها محاسبه می‌شود تا SciPy لازم نباشد. هم‌بستگی نزدیک صفر فقط نشان می‌دهد که در این مجموعه، رابطهٔ یکنواخت روشنی بین حجم همان ماه و میانهٔ زمان بسته‌شدن دیده نمی‌شود؛ این آزمون، تحلیل علّی یا آزمون معناداری نیست.

In [ ]:
monthly_duration = (
    records.dropna(subset=["borough"])
    .groupby(["borough", "month"])
    .agg(request_count=("unique_key", "size"), median_closure_days=("closure_days", "median"))
    .dropna(subset=["median_closure_days"])
)
spearman_rho = monthly_duration["request_count"].rank().corr(
    monthly_duration["median_closure_days"].rank()
)
fig, ax = plt.subplots(figsize=(9, 6))
for borough, group in monthly_duration.groupby(level=0):
    ax.scatter(group["request_count"], group["median_closure_days"], label=borough, alpha=0.8)
ax.text(0.03, 0.97, f"Spearman rho = {spearman_rho:.3f}; borough-months = {len(monthly_duration)}", transform=ax.transAxes, va="top")
ax.set(title="Monthly request volume vs. median closure duration", xlabel="Requests in borough-month", ylabel="Median closure days")
ax.legend(title="311 borough", ncols=2)
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## ۸. یافته‌ها و محدودیت‌های تفسیر

- **الگوی زمانی:** اوج تعداد درخواست‌ها در ژانویهٔ ۲۰۲۵ (۶۷٬۲۲۹) و کمینه در اوت ۲۰۲۴ (۳٬۱۰۵) است. این تفاوت با فصل گرمایش سازگار است، اما فقط ۱۸ ماه مشاهده شده و تغییرات منبع نیز باید در نظر گرفته شود.
- **تفاوت جغرافیایی:** Bronx با ۱۵۵٬۰۴۵ درخواست بیشترین شمار را دارد. نرخ تجمعی ۱۸ماههٔ آن حدود ۱٬۰۵۹ درخواست در هر ۱۰٬۰۰۰ نفر مرجع است؛ در Staten Island حدود ۹۵ است. این‌ها رخدادهای درخواست‌اند، نه خانوارهای یکتا، و جمعیت ZIP/ZCTA تقریبی است.
- **دنبالهٔ زمان بسته‌شدن:** از ۴۲۷٬۴۵۸ رکورد، ۴۲۳٬۰۷۷ مورد بسته شده‌اند و میانهٔ زمان بسته‌شدن ۱٫۵۷۳ روز است. صدک ۹۹ حدود ۵٫۰۳۵ روز و بیشینه ۵۵۲٫۲۳۲ روز است. ۶٬۸۹۹ مقدار از مرز ۱٫۵ برابر IQR فراتر می‌روند؛ فقط به‌دلیل طولانی بودن حذف نشده‌اند.
- **تفاوت میان boroughها:** میانهٔ Bronx حدود ۱٫۳۱۷ روز و Manhattan حدود ۱٫۸۹۶ روز است. ترکیب و پیچیدگی پرونده‌ها در این مقایسه کنترل نشده است.
- **تطبیق جمعیت:** ۴۲۷٬۴۰۵ رکورد به ZIP مرجع پیوسته و ۵۳ مورد بدون تطبیق مانده‌اند؛ ۸۳۵ مورد اختلاف borough میان 311 و نگاشت مرجع دارند. اختلاف‌ها به‌جای جایگزین‌کردن borough منبع، برای بازبینی نگه داشته شده‌اند.
- **حجم در برابر زمان بسته‌شدن:** هم‌بستگی رتبه‌ای میان شمار ماهانه و میانهٔ زمان بسته‌شدن در ۹۰ جفت borough-ماه حدود ۰٫۰۱۳ است؛ رابطهٔ یکنواخت قابل‌توجهی در این برش دیده نمی‌شود.

### محدودیت بازتولید

این اجرای تحلیل از خروجی‌های محلی بازتولیدشده استفاده می‌کند و درخواست شبکه‌ای نمی‌فرستد. نه صفحهٔ خام و checkpoint کامل در `data/raw/311/` موجودند. اجرای آنلاین کارخانه در درخواست متادیتای schema با HTTP 403 متوقف شد؛ replay آفلاین یک `run_manifest.json` ساخت که `source_schema_validation: skipped_offline` و `request_count: 0` را ثبت می‌کند. این manifest جایگزین تأیید schema زنده یا دانلود آنلاین نیست.